# جلسهٔ ۰۴ — جست‌وجوی خصمانه و بازی‌ها

**سطح متوسط | قابل اجرا روی CPU | آفلاین | بدون نیاز به کلید API**

در جست‌وجوی خصمانه (Adversarial Search)، عامل باید هنگام برنامه‌ریزی، تصمیم‌های یک حریف راهبردی را نیز در نظر بگیرد. این ویژگی آن را از جست‌وجوی معمول برای یافتن کوتاه‌ترین مسیر متمایز می‌کند.

### اهداف یادگیری
- مفهوم **وضعیت بازی (Game State)** و **تابع مطلوبیت (Utility)** را توضیح دهید و برای آن‌ها مثال بزنید.
- الگوریتم **مینیمکس (Minimax)** را توضیح دهید و در یک بازی به کار ببرید.
- روش **هرس آلفا–بتا (Alpha-Beta Pruning)** را توضیح دهید و اثر آن را بر جست‌وجو بررسی کنید.
- مفهوم **ارزیابی ابتکاری (Heuristic Evaluation)** را توضیح دهید و کاربرد آن را در بازی‌هایی با فضای جست‌وجوی بزرگ بشناسید.

ابتدا **README.md** اصلی دوره را بخوانید. پیش از اجرای هر سلول، نتیجهٔ آن را پیش‌بینی کنید.

## مثال حل‌شده

فرض کنید بازیکن X دو حرکت ممکن دارد و بازیکن O برای هر حرکت می‌تواند یکی از دو پاسخ را انتخاب کند.

برای هر حرکت X، کمترین امتیاز حاصل از پاسخ‌های O را حساب کنید. سپس حرکتی را انتخاب کنید که بیشترین مقدار را در میان این کمینه‌ها دارد.

**پرسش کلاسی:** در درخت بازی، کدام بازیکن گره‌های کمینه‌ساز (MIN) را کنترل می‌کند؟

## قوانین صفحهٔ بازی

بازی دوز (Tic-Tac-Toe) را با صفحه‌ای شامل ۹ خانه و هر ۸ خط ممکن برای پیروزی نمایش دهید.

- `LINES` سه سطر، سه ستون و دو قطر برنده را مشخص می‌کند.
- تابع `result` در صورت پیروزی X مقدار ۱، در صورت پیروزی O مقدار ۱-، در حالت مساوی مقدار صفر و در صورت ادامه داشتن بازی مقدار `None` را برمی‌گرداند.
- تابع `moves` خانه‌های خالی و در نتیجه حرکت‌های مجاز را مشخص می‌کند.
- تابع `moved` یک نسخهٔ جدید از صفحه با حرکت انتخاب‌شده ایجاد می‌کند، بدون تغییر صفحهٔ قبلی.

خانه‌های صفحه در کد از صفر تا هشت شماره‌گذاری شده‌اند.

In [ ]:
LINES=[(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
def result(b):
    for a,c,d in LINES:
        if b[a] and b[a]==b[c]==b[d]:return 1 if b[a]=="X" else -1
    return 0 if all(b) else None
def moves(b):return [i for i,x in enumerate(b) if not x]
def moved(b,i,p):
    out=b.copy();out[i]=p;return out
print("حرکت‌های مجاز در صفحهٔ خالی:",moves([""]*9))

## الگوریتم مینیمکس (Minimax)

همهٔ امتیازها از دید بازیکن X تعریف می‌شوند. بنابراین X بازیکن بیشینه‌ساز (MAX) و O بازیکن کمینه‌ساز (MIN) است.

الگوریتم به‌صورت بازگشتی حرکت‌های مجاز را بررسی می‌کند تا به وضعیت پایانی برسد. در نوبت X بیشترین امتیاز و در نوبت O کمترین امتیاز انتخاب می‌شود.

در این مثال، امتیاز حرکت‌های ممکن X را روی صفحهٔ مشخص‌شده حساب کنید. قبل از اجرای کد پیش‌بینی کنید کدام حرکت بهترین نتیجه را خواهد داشت.

In [ ]:
def minimax(b,turn):
    terminal=result(b)
    if terminal is not None:return terminal
    other="O" if turn=="X" else "X"
    values=[minimax(moved(b,i,turn),other) for i in moves(b)]
    return max(values) if turn=="X" else min(values)
board=["X","O","X","O","X","","","","O"]
scores={i:minimax(moved(board,i,"X"),"O") for i in moves(board)}
print("امتیاز حرکت‌ها:",scores)
assert max(scores,key=scores.get)==6 and scores[6]==1

## هرس آلفا–بتا (Alpha-Beta Pruning)

یک بهینه‌سازی درست باید مقدار نهایی بازی را که مینیمکس محاسبه می‌کند، حفظ کند.

- `alpha` بهترین کران پایین شناخته‌شده برای بازیکن بیشینه‌ساز است.
- `beta` بهترین کران بالای شناخته‌شده برای بازیکن کمینه‌ساز است.
- وقتی `alpha >= beta` شود، ادامهٔ بررسی آن شاخه نمی‌تواند انتخاب بهینه را تغییر دهد و می‌توان آن را هرس کرد.

مقدار بازگشتی آلفا–بتا را با مقدار مینیمکس روی همان صفحه مقایسه کنید.

In [ ]:
def alphabeta(b,turn,alpha=-2,beta=2):
    t=result(b)
    if t is not None:return t
    if turn=="X":
        value=-2
        for i in moves(b):
            value=max(value,alphabeta(moved(b,i,"X"),"O",alpha,beta))
            alpha=max(alpha,value)
            if alpha>=beta:break
    else:
        value=2
        for i in moves(b):
            value=min(value,alphabeta(moved(b,i,"O"),"X",alpha,beta))
            beta=min(beta,value)
            if alpha>=beta:break
    return value
print("آلفا–بتا:",alphabeta(board,"X"))
assert alphabeta(board,"X")==minimax(board,"X")

## تمرین

صفحه‌ای طراحی کنید که در آن یک حرکت بلافاصله باعث پیروزی شود و بررسی کنید که مینیمکس آن را به‌عنوان یک حرکت برنده شناسایی می‌کند.

**تکلیف:** تعداد وضعیت‌های بررسی‌شده را در مینیمکس و آلفا–بتا بشمارید و با هم مقایسه کنید.

**برداشت نادرست رایج:** مینیمکس فرض می‌کند حریف راهبردی عمل می‌کند، نه اینکه پاسخ خود را به‌صورت تصادفی انتخاب کند.

نوت‌بوک را در یک کرنل تازه و به‌ترتیب اجرا کنید و دست‌کم یکی از محدودیت‌های روش خود را توضیح دهید.